# Assess knowledge answers

**Goal:** Distinguish multiple-choice correctness, free-text agreement and criteria coverage, including perfect retrieval.

Run cells from top to bottom. Default cells work offline; model execution is an explicit opt-in and writes only to ignored `outputs/`.

## 1. Set up paths

Find the repository and load the analysis helpers.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data/final").is_dir(), "Run from the repository or notebooks folder"
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from sleepinn_study.io import read_json, read_jsonl, output_directory
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})


## 2. Read the scoring rules

MCQs are assessed against the keyed option, with ambiguous formats retained for adjudication. Free-text primary scores use strict reference agreement. Criteria coverage gives covered components 1, partial components 0.5 and missing/contradicted components 0; an ungradable component makes the item missing. Invalid outputs are not silently converted to incorrect answers.

In [ ]:
from sleepinn_study.scoring import parse_mcq, criteria_coverage
display(read_json(ROOT / "data/config/knowledge_scoring_rubric.json"))
print("Example MCQ:", parse_mcq("Answer: B. Explanation.", list("ABCD")))
print("Example criteria coverage:", criteria_coverage(["covered", "partial", "missing"]))

## 3. Inspect the semantic assessment prompt

The scoring code is visible here. No judge is called. Original no-RAG and selected-RAG scores remain from Gemini; perfect-retrieval semantic answers were assessed with GLM 5.3 Flash. That judge difference limits attribution of perfect-retrieval gains to retrieval alone.

In [ ]:
import inspect
from sleepinn_study import knowledge_judge
print(inspect.getsource(knowledge_judge.request))
scores = pd.read_csv(ROOT / "results/analysis/knowledge/answer_scores.csv")
assert len(scores) == 29160
display(scores.groupby(["item_type", "mode"]).primary_score.agg(["count", "mean"]))
from sleepinn_study.knowledge_glm_judge import text_request
print(inspect.getsource(text_request))


## 4. Compare all three conditions by question type

Keep the endpoints separate: a single pooled score would mix accuracy with component coverage. Bars show means over available question scores, with coverage reported in the table above.

In [ ]:
summary = scores.groupby(["item_type", "mode"]).primary_score.mean().unstack().reindex(columns=["no_rag", "with_rag", "perfect_retrieval"])
ax = (summary * 100).plot.bar(rot=0, figsize=(9, 4), color=["#a3b8c2", "#327c9d", "#d88735"])
ax.set(xlabel="Question type", ylabel="Primary score (%)", ylim=(0,100))
plt.tight_layout()
plt.savefig(output_directory("figures") / "knowledge_three_conditions.png")
plt.show()

## 5. Explore source and disorder topics

Join by item ID, not row position. The public bank retains source/topic labels and all original answer text.

In [ ]:
bank = pd.DataFrame(read_jsonl(ROOT / "data/final/knowledge.jsonl"))
topic = scores.merge(bank[["item_id", "topic", "section_label"]], on="item_id", validate="many_to_one")
display(topic.groupby(["source_pdf", "item_type", "mode"]).primary_score.agg(["count", "mean"]))
display(topic.groupby(["topic", "item_type", "mode"]).primary_score.agg(["count", "mean"]))